# Import Libraries

In [1]:
import sys
from pathlib import Path
repo_root = Path.cwd().resolve()
src_path = (repo_root.parent / "src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(src_path)

from tdims import tdims_ext
import pandas as pd

/Users/lisahamada/materials/models/tdims/src


# Load Data

In [4]:
data_df = pd.read_csv('../data/cmpCl3_200.csv')

smi_col = 'SMILES'
target_prop = 'def_EmAbs'

sm_list = list(data_df[smi_col].values)
y = list(data_df[target_prop].values)

In [5]:
print(f'Data size: {data_df.shape}')
data_df.head(2)

Data size: (200, 6)


,SMILES,Solvent,Emission max (nm),Abs,Solvent_id,def_EmAbs
0,C[Si](C)(C)C#CC(C#C[Si](C)(C)C)=Cc1ccc(C=C(C#C...,ClC(Cl)Cl,475.0,458.0,1,17.0
1,C(=N/c1ccc(-c2cn3nc(-c4cccs4)sc3n2)cc1)\c1c2cc...,ClC(Cl)Cl,526.0,380.0,1,146.0


# Feature Embeddings

### Full embeddings

In [14]:
radius = 1
fragment = False
func_dis = -1
func_dup = max

emb, key_all = tdims_ext.get_representation(sm_list, radius=radius, func_dis=func_dis, func_merge=func_dup, fragment_set=fragment, atom_set=True, fingerprint_set=True, display=True)

Full embedding shape: (200, 3347)
Execution time for full embedding:  2.047978 sec


In [15]:
key_all[:5]

['C[Si](C)(C)C & C[Si](C)(C)C',
 'C[Si](C)(C)C & C#C[SiH3]',
 'C[Si](C)(C)C & C=C(C)C',
 'C[Si](C)(C)C & cc(C)s',
 'C[Si](C)(C)C & C#CC']

In [16]:
print(f"length of the key: {len(key_all)}")
print(f"shape of the emb: {emb.shape}")

length of the key: 3347
shape of the emb: (200, 3347)


In [17]:
df_fv = pd.DataFrame(columns=key_all, data=emb)
display(df_fv.head())

,C[Si](C)(C)C & C[Si](C)(C)C,C[Si](C)(C)C & C#C[SiH3],C[Si](C)(C)C & C=C(C)C,C[Si](C)(C)C & cc(C)s,C[Si](C)(C)C & C#CC,C[Si](C)(C)C & C=CC,C[Si](C)(C)C & ccc,C[Si](C)(C)C & csc,C[Si](C)(C)C & S,C#C[SiH3] & C#C[SiH3],...,cn(c)c & cc(c)o,cn(c)c & CCN,cn(c)c & coc,cn(c)c & O,cc(n)n & cc(c)o,cc(n)n & CCN,cc(n)n & coc,cc(n)n & O,cc(c)o & ccn,ccn & coc
0,0.147059,0.652174,0.273973,0.176991,0.416667,0.227273,0.15625,0.15625,0.15625,0.25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Embeddings w/ feature selection including parameter optimization

In [4]:
radius = 2
func_dis = -1
func_dup = max
fragment = False

x_slc, key_slc, key_all = tdims_ext.get_representation_with_fs_selection(sm_list, y, radius=radius, func_dis=func_dis, func_merge=func_dup, fragment_set=fragment, atom_set=True, fingerprint_set=True)


Full embedding shape: (200, 17439)
Execution time for full embedding:  3.875421 sec

Feature were selected from (200, 17439) to (200, 311)
Execution time for feature selection:  6.501764 sec


In [5]:
len(key_slc)

311

# Regression

In [6]:
result = tdims_ext.run_tdims_regression_cv(
    sm_list=sm_list,
    y=y,
    radius=2,
    func_dis=-1,
    func_merge=max,
    fragment_set=False,
    cv=3,
    n_repeats=1,
    inner_cv=3,
    random_state=1,
    model_name="lasso",
    mode="quick",
    use_feature_selection=True,
    fs_alpha=0.0001,
    n_jobs=-1,
    verbose=0,
)

Full embedding shape: (200, 17439)
Execution time for full embedding:  3.795784 sec

Full descriptor matrix shape: (200, 17439)

===== TDiMS regression CV result =====
Model family        : lasso
Mode                : quick
Descriptor shape    : (200, 17439)
Use FS              : True
Outer CV            : RepeatedKFold(n_splits=3, n_repeats=1)
Inner CV            : KFold(n_splits=3, shuffle=True)
R2 scores           : [0.3627 0.4478 0.4682]
Mean R2 +/- std     : 0.4262 +/- 0.0560
Execution time      : 21.241267 sec


# Feature importance analysis (SHAP)

In [7]:
import shap
import numpy as np
from sklearn.linear_model import Lasso

In [10]:
optimized_param = dict(result["final_best_params"])
x_slc = np.asarray(result["final_X_selected"])
key_slc = result["final_selected_keys"]
y_used = np.asarray(result["final_y"])

estimator = Lasso(
    random_state=0,
    max_iter=100000,
    **optimized_param
)

estimate = estimator.fit(x_slc, y_used)
explainer = shap.LinearExplainer(estimate, shap.maskers.Independent(np.array(x_slc)))

shap_values = explainer(x_slc)
shap_val_X = shap_values.values
shap_val_absX = np.abs(shap_val_X)

df_shap_val_abs = pd.DataFrame(columns=key_slc, data = shap_val_absX)

In [11]:
df_shap_val_abs.head()

,C=C(C)C#C[SiH3] & C=C(C)C#C[SiH3],C=C(C)C#C[SiH3] & C#CC(C#C)=CC,Cc1ccc(C)s1 & S,ccc(c(c)C)c(c)c & cccc(c)C,ccc(cc)c(c)c & ccc(cc)c(c)c,ccc(cc)c(c)c & cccc(c)C,ccc(cc)c(c)c & ccccc,cc(c)cc(c)c & ccccc,C=Nc(cc)cc & cc(c)C=NC,C=Nc(cc)cc & cc(c)N=CC,...,C#Cc1cccs1 & S,ccc(cc)-c(c)o & cc(c)-c1ccco1,cc(o)C=C(C)C & CC=C(C#N)C#N,cc(c)ncn & N,cc(O)c(OC)c(c)c & O,O & P,cc(c)OC(C)(C)C & O,cc(c)N1CCCC1 & N,cc[n+](cc)CC & N,Cc1ncc(N)[se]1 & Cc1c[se]c(C)n1
0,4.609072,31.879876,4.723151,0.092641,0.385854,0.104213,0.534590,0.371567,0.00000,0.000000,...,0.0,0.0,0.0,0.0,0.003649,0.0,0.0,0.0,0.005777,0.0
1,0.000000,0.000000,0.146077,0.840456,3.547658,0.677668,4.345717,14.536973,2.64559,18.268344,...,0.0,0.0,0.0,0.0,0.003649,0.0,0.0,0.0,0.005777,0.0
2,0.000000,0.000000,0.146077,0.092641,0.385854,0.104213,0.534590,0.371567,0.00000,0.000000,...,0.0,0.0,0.0,0.0,0.003649,0.0,0.0,0.0,0.005777,0.0
3,0.000000,0.000000,0.146077,0.092641,0.385854,0.104213,0.534590,0.371567,0.00000,0.000000,...,0.0,0.0,0.0,0.0,0.003649,0.0,0.0,0.0,0.005777,0.0
4,0.000000,0.000000,0.146077,0.092641,4.302030,0.104213,4.121335,21.876562,0.00000,0.000000,...,0.0,0.0,0.0,0.0,0.003649,0.0,0.0,0.0,0.005777,0.0


# TDiMS hyperparameter optimization and evaluation

This section demonstrates the nested cross-validation pipeline for TDiMS.  
Both model hyperparameters and TDiMS descriptor parameters are optimized in the inner loop, and performance is evaluated in the outer loop.  
Because this can be computationally expensive, full runs are recommended in a Python script rather than in the notebook.

In [ ]:
from experiment_sample import main

main(
    database="./data/cmpCl3_200.csv",
    prop="def_EmAbs",
    desc_name="TDiMS",
    outer_random_state=0,
    outer_n_repeats=1,
    n_jobs=-1,
    out_dir_sub="experimentcode",
    etc="notebook_example",
    save_cv_results=True,
    save_joblib=False,
    mode="quick",
    verbose=0
)